# 07 — Pairwise evaluation with a Kaggle model judge (Task 7)

Create this notebook as a **utility benchmark** (https://www.kaggle.com/benchmarks/tasks/new) so
`kaggle_benchmarks` and the model proxy are available. No GPU is needed. **Add Input** → saved output of
notebook 06. Judging reads only the saved pairwise JSONL files; no model outputs are regenerated.

Reliability (manual 13.2): 5 pairs per request, ≥5 s between attempts, up to 3 retries, failures logged to
`*_failures.jsonl`, judgments checkpointed after every batch, `--reuse_judgments` resumes a partial run.

In [ ]:
# Clone (or update) the project repo into /kaggle/working. Notebook settings: Internet ON.
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {cmd}")

In [ ]:
import kaggle_benchmarks as kbench
JUDGE = "google/gemini-3.6-flash"   # if unavailable, pick another from the list below and record it in the report
print(sorted(kbench.llms))

In [ ]:
# Locate artifacts from earlier notebooks attached under /kaggle/input (Add Input -> your notebook output).
import glob, os
def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Attach the notebook output that contains {pattern}"
    print(pattern, "->", hits[0])
    return hits[0]

In [ ]:
import shutil
for name in ("dpo_uf_vs_sft_pairs.jsonl", "dpo_hh_vs_sft_pairs.jsonl", "lora_vs_base_pairs.jsonl"):
    os.makedirs("outputs/eval", exist_ok=True)
    shutil.copy(find_input(f"eval/{name}"), f"outputs/eval/{name}")
# Resume support: also copy any judgments saved by a previous version of this notebook, if attached.
for pattern in ("eval/*_judged.jsonl", "eval/*_judged_failures.jsonl"):
    for path in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        shutil.copy(path, "outputs/eval/")
!ls -l outputs/eval

In [ ]:
# Small smoke run first (2 pairs) to confirm the judge responds and parses.
sh("python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl "
   f"--output_path outputs/eval/smoke_judged.jsonl --judge_model {JUDGE} --limit 2 "
   "--batch_size 5 --sleep_seconds 5 --max_retries 3")
!cat outputs/eval/smoke_judged_raw.jsonl | head -c 2000

In [ ]:
for name in ("dpo_uf_vs_sft", "dpo_hh_vs_sft", "lora_vs_base"):
    sh(f"python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/{name}_pairs.jsonl "
       f"--output_path outputs/eval/{name}_judged.jsonl --judge_model {JUDGE} "
       "--batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments")

In [ ]:
!rm -f outputs/eval/smoke_judged*
!python scripts/collect_results.py --eval_dir outputs/eval --output outputs/eval/results.md

In [ ]:
!cd outputs && zip -q -r /kaggle/working/judge_outputs.zip eval
!ls -lh /kaggle/working/judge_outputs.zip

**Save Version** when judging completes. If some batches failed, re-run the judging cell; completed pairs are reused.